# Day 1 · Exercise 2: Which factory should ship to which DC?

**What you'll be able to say by the end**
1. "Shipping on the cheapest route first cost **$6,000 a week more** than the best plan, because it used up a factory that another DC needed more."
2. "One more pair at New Jersey costs **$3.70**, even though the cheapest open route there is $3.60. That's the knock-on effect of moving other shipments around."
3. "A Long Beach hub saves **$20,500 a week**. Each extra pair of hub capacity is worth **$0.60** until about 35,000 pairs, then less."

**New ideas today:** the *transportation problem*, *reduced cost*, a *hub* (transshipment) with a "what goes in must come out" rule, and a peek at how the solver actually finds the answer.

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


# Part A: Ship directly from factories to DCs

## 1. The story

Three factories in Asia (Vietnam, Indonesia, China) make the same shoe. Five US **distribution centers (DCs)** each need a set number of pairs this week. Shipping one pair costs a different amount on each route.

**The question:** which factory should send how many pairs to which DC, so every DC gets what it needs at the lowest total shipping cost?

In [3]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

FACTORIES = ["Vietnam", "Indonesia", "China"]
DCS = ["Los Angeles", "Portland", "Memphis", "New Jersey", "Chicago"]

supply = {"Vietnam": 50_000, "Indonesia": 30_000, "China": 25_000}   # pairs each factory can make this week
demand = {"Los Angeles": 20_000, "Portland": 15_000, "Memphis": 25_000,
          "New Jersey": 25_000, "Chicago": 15_000}                   # pairs each DC needs this week

ship_cost = pd.DataFrame(   # $ per pair, factory -> DC directly
    [[2.0, 2.3, 3.4, 3.9, 3.5],
     [2.2, 2.5, 3.3, 3.6, 3.4],
     [1.8, 2.0, 3.6, 4.2, 3.7]],
    index=FACTORIES, columns=DCS,
)

table = ship_cost.copy()
table["can make"] = pd.Series(supply)
table.loc["needs"] = pd.Series(demand)
table.columns.name = "$ per pair"
table

$ per pair,Los Angeles,Portland,Memphis,New Jersey,Chicago,can make
Vietnam,2.0,2.3,3.4,3.9,3.5,50000.0
Indonesia,2.2,2.5,3.3,3.6,3.4,30000.0
China,1.8,2.0,3.6,4.2,3.7,25000.0
needs,20000.0,15000.0,25000.0,25000.0,15000.0,NaN


**How to read it:** each number in the middle is the cost to ship one pair on that route. For example, Vietnam → Memphis costs $3.40 a pair. The last column is what each factory can make; the bottom row is what each DC needs.

A few things to notice before solving:
- **China** is the cheapest way to reach the West Coast (LA $1.80, Portland $2.00), but it's the smallest factory.
- **Indonesia** is the cheapest way to reach **New Jersey** ($3.60) and **Memphis** ($3.30).
- **Vietnam** is the biggest factory and is never the cheapest anywhere, but it's never far off.

## 2. Can the factories cover all the DCs?

In [4]:
print(f"Factories can make: {sum(supply.values()):,} pairs")
print(f"DCs need:           {sum(demand.values()):,} pairs")
print(f"Spare:              {sum(supply.values()) - sum(demand.values()):,} pairs")

Factories can make: 105,000 pairs
DCs need:           100,000 pairs
Spare:              5,000 pairs


Yes, with 5,000 pairs to spare. So the question isn't *whether* we can serve everyone. It's **which routes to use** so the total cost is lowest.

## 3. Solve it by hand: cheapest route first

The obvious rule: find the cheapest route in the whole table, ship as much as you can on it, cross off whatever is used up, and repeat.

In [5]:
left_supply, left_demand = dict(supply), dict(demand)
hand = pd.DataFrame(0.0, index=FACTORIES, columns=DCS)

for (f, d), cost in ship_cost.stack().sort_values().items():
    qty = min(left_supply[f], left_demand[d])
    if qty > 0:
        hand.loc[f, d] = qty
        left_supply[f] -= qty
        left_demand[d] -= qty
        print(f"${cost:.2f}  {f:<9} -> {d:<11} ship {qty:>6,.0f}   ({f} has {left_supply[f]:>6,.0f} left)")

hand_cost = (hand * ship_cost).to_numpy().sum()
print(f"\nCost by hand: ${hand_cost:,.0f}")

$1.80  China     -> Los Angeles ship 20,000   (China has  5,000 left)
$2.00  China     -> Portland    ship  5,000   (China has      0 left)
$2.30  Vietnam   -> Portland    ship 10,000   (Vietnam has 40,000 left)
$3.30  Indonesia -> Memphis     ship 25,000   (Indonesia has  5,000 left)
$3.40  Indonesia -> Chicago     ship  5,000   (Indonesia has      0 left)
$3.50  Vietnam   -> Chicago     ship 10,000   (Vietnam has 30,000 left)
$3.90  Vietnam   -> New Jersey  ship 25,000   (Vietnam has  5,000 left)

Cost by hand: $301,000


Look at the **last line** of the list: New Jersey ends up being served from **Vietnam at $3.90**, the most expensive route anyone uses.

Why? Early on, the rule happily used Indonesia for Memphis ($3.30). By the time New Jersey's turn came, Indonesia, its cheapest source, had already been used up elsewhere. The rule only looks at one route at a time, so it can't see that coming.

## 4. Let a solver do it

This is a **linear program (LP)**, just like the shoe factory in exercise 1. It has the same three pieces:

| Piece | Plain words | Here |
|---|---|---|
| **Decision variables** | The numbers we choose | Pairs shipped on each of the 15 routes (3 factories × 5 DCs) |
| **Objective** | What we want as small as possible | Total shipping cost = Σ (cost per pair × pairs) over all routes |
| **Constraints** | Rules we can't break | Each factory ships **at most** what it can make (≤). Each DC gets **at least** what it needs (≥). No negative shipments |

This shape (sources with limited supply, destinations with demand, a cost per route) is so common that it has a name: the **transportation problem**.

In [6]:
import pulp

def var_name(*parts):
    return "_".join(parts).replace(" ", "_")

def solve_direct():
    model = pulp.LpProblem("shipping", pulp.LpMinimize)

    # Decision variables: pairs shipped on each route
    ship = {(f, d): model.add_variable(var_name("ship", f, d), lowBound=0) for f in FACTORIES for d in DCS}

    # Objective: total shipping cost
    model += pulp.lpSum(ship_cost.loc[f, d] * ship[f, d] for f in FACTORIES for d in DCS)

    # Constraint 1: a factory can't ship more than it makes
    for f in FACTORIES:
        model += pulp.lpSum(ship[f, d] for d in DCS) <= supply[f], var_name("supply", f)

    # Constraint 2: every DC gets at least what it needs
    for d in DCS:
        model += pulp.lpSum(ship[f, d] for f in FACTORIES) >= demand[d], var_name("demand", d)

    stats = model.solve(pulp.HiGHS(msg=False))
    assert stats.status == pulp.LpSolveStatus.Optimal, stats.status
    return model, ship


model_a, ship_a = solve_direct()
cost_a = pulp.value(model_a.objective)
plan_a = pd.DataFrame({d: {f: ship_a[f, d].varValue for f in FACTORIES} for d in DCS})

print(f"Cost by hand: ${hand_cost:,.0f}")
print(f"Cost solver:  ${cost_a:,.0f}   -> solver saves ${hand_cost - cost_a:,.0f} every week")
plan_a

Cost by hand: $301,000
Cost solver:  $295,000   -> solver saves $6,000 every week


,Los Angeles,Portland,Memphis,New Jersey,Chicago
Vietnam,10000.0,0.0,25000.0,0.0,10000.0
Indonesia,0.0,0.0,0.0,25000.0,5000.0
China,10000.0,15000.0,0.0,0.0,0.0


**What the solver did differently:** it kept **Indonesia for New Jersey** (25,000 pairs at $3.60), because nobody else can reach New Jersey cheaply. Memphis goes to Vietnam instead ($3.40, only $0.10 worse than Indonesia). It gave up a little on Memphis to save a lot on New Jersey ($3.90 → $3.60).

That's the key thing an LP does that a one-route-at-a-time rule can't: it weighs **every route against every other route at once**.

Here is the solver's plan as a picture. The width of each band is the number of pairs:

In [7]:
import plotly.graph_objects as go

def flow_picture(flows, title):
    """flows: list of (from, to, pairs). Draws a Sankey diagram."""
    nodes = list(dict.fromkeys([a for a, _, _ in flows] + [b for _, b, _ in flows]))
    idx = {n: i for i, n in enumerate(nodes)}
    fig = go.Figure(go.Sankey(
        node=dict(label=nodes, pad=18, thickness=16),
        link=dict(source=[idx[a] for a, _, _ in flows], target=[idx[b] for _, b, _ in flows],
                  value=[q for _, _, q in flows],
                  hovertemplate="%{source.label} -> %{target.label}<br>%{value:,.0f} pairs<extra></extra>"),
    ))
    fig.update_layout(title=title, height=420, width=900, template="plotly_white")
    fig.show()

flow_picture([(f, d, q) for (f, d), v in ship_a.items() if (q := v.varValue) > 1e-6],
             f"Solver plan, direct shipping only: ${cost_a:,.0f}/week")

## 5. Shadow prices: what does one more pair cost?

Same idea as exercise 1, but now we're **minimizing cost**, so the shadow prices read as costs and savings:

- **DC rows:** if this DC needed **one more pair**, how much would total cost go up?
- **Factory rows:** if this factory could make **one more pair**, how much would total cost go down?

In [8]:
why = {
    "supply_Vietnam":     "Already has 5,000 spare pairs, so one more saves nothing",
    "supply_Indonesia":   "One more pair to Chicago ($3.40) replaces a Vietnam -> Chicago pair ($3.50): saves $0.10",
    "supply_China":       "One more pair to LA ($1.80) replaces a Vietnam -> LA pair ($2.00): saves $0.20",
    "demand_Los_Angeles": "Vietnam has spare pairs: Vietnam -> LA = $2.00",
    "demand_Portland":    "China -> Portland is $2.00, but China is full. Move one China pair off LA and cover LA from Vietnam (+$0.20). Total $2.20",
    "demand_Memphis":     "Vietnam has spare pairs: Vietnam -> Memphis = $3.40",
    "demand_New_Jersey":  "Indonesia -> NJ is $3.60, but Indonesia is full. Move one Indonesia pair off Chicago and cover Chicago from Vietnam (+$0.10). Total $3.70",
    "demand_Chicago":     "Vietnam has spare pairs: Vietnam -> Chicago = $3.50",
}
pd.DataFrame(
    [{"constraint": c.name, "left over": round(c.slack) + 0, "$ per extra pair": round(abs(c.pi), 2), "why": why[c.name]}
     for c in model_a.constraints()]
).set_index("constraint")

,left over,$ per extra pair,why
constraint,,,
supply_Vietnam,5000,0.0,"Already has 5,000 spare pairs, so one more saves nothing"
supply_Indonesia,0,0.1,One more pair to Chicago ($3.40) replaces a Vietnam -> Chicago pair ($3.50): saves $0.10
supply_China,0,0.2,One more pair to LA ($1.80) replaces a Vietnam -> LA pair ($2.00): saves $0.20
demand_Los_Angeles,0,2.0,Vietnam has spare pairs: Vietnam -> LA = $2.00
demand_Portland,0,2.2,"China -> Portland is $2.00, but China is full. Move one China pair off LA and cover LA from Vietnam (+$0.20). Total $2.20"
demand_Memphis,0,3.4,Vietnam has spare pairs: Vietnam -> Memphis = $3.40
demand_New_Jersey,0,3.7,"Indonesia -> NJ is $3.60, but Indonesia is full. Move one Indonesia pair off Chicago and cover Chicago from Vietnam (+$0.10). Total $3.70"
demand_Chicago,0,3.5,Vietnam has spare pairs: Vietnam -> Chicago = $3.50


**The interesting rows are Portland and New Jersey.** Their cheapest route costs $2.00 and $3.60, but one more pair costs **$2.20** and **$3.70**. The cheapest factory is already full, so serving one more pair means shuffling another shipment, and that shuffle costs a bit extra. A shadow price includes all of those knock-on effects. That's why it's more reliable than looking up the cheapest route.

**Why are some numbers negative inside the solver?** PuLP reports a factory's value as a negative number, because one more pair of capacity *lowers* cost. We show the size of it ("saves $0.20") with `abs()`. The meaning is the same.

**What a planner does with this:**
- If New Jersey asks for extra pairs this week, quote them **$3.70 a pair**, not $3.60.
- Adding capacity in **China** saves $0.20 a pair. Adding it in **Vietnam** saves nothing, because Vietnam already has spare.

## 6. Reduced cost: how much cheaper must an unused route get?

The solver used only 7 of the 15 routes. For each unused route, the **reduced cost** says how much its price would have to drop before the solver would start using it.

There's a simple way to see it, using the shadow prices from step 5:

> **reduced cost = route cost − (cost of 1 more pair at the DC − saving from 1 more pair at the factory)**

Example: **Vietnam → Portland** costs $2.30. One more pair at Portland costs $2.20 (step 5), and Vietnam's spare capacity is worth $0. So the reduced cost is 2.30 − (2.20 − 0) = **$0.10**. Get that route $0.10 cheaper and it's worth using.

In [9]:
unused = pd.DataFrame(
    [{"route": f"{f} -> {d}", "cost now": ship_cost.loc[f, d], "must get cheaper by": round(v.dj, 2),
      "worth using below": round(ship_cost.loc[f, d] - v.dj, 2)}
     for (f, d), v in ship_a.items() if v.varValue < 1e-6]
).set_index("route").sort_values("must get cheaper by")
unused

,cost now,must get cheaper by,worth using below
route,,,
Indonesia -> Memphis,3.3,0.0,3.3
Vietnam -> Portland,2.3,0.1,2.2
Vietnam -> New Jersey,3.9,0.2,3.7
Indonesia -> Los Angeles,2.2,0.3,1.9
Indonesia -> Portland,2.5,0.4,2.1
China -> Memphis,3.6,0.4,3.2
China -> Chicago,3.7,0.4,3.3
China -> New Jersey,4.2,0.7,3.5


**Indonesia → Memphis shows $0.00.** That means using it would cost exactly the same: there's a second plan that ties for cheapest. (Sending Indonesia pairs to Memphis instead of Chicago, and Vietnam pairs to Chicago instead of Memphis, comes out even.) Ties like this are common in real problems. They're a hint that a planner has some free choice there.

**What a planner does with this:** when a carrier offers a new rate, compare it to "worth using below". If Vietnam → Portland drops under **$2.20**, the plan should change. Otherwise it shouldn't.

# Part B: Add a hub

## 7. A hub at the Port of Long Beach

Now there's a second option: send shoes by ocean to a **hub at Long Beach** (California), then forward them by rail or truck to any DC. Ocean freight to the hub is cheap, so going through the hub is often cheaper than shipping directly. But the hub can only handle **30,000 pairs a week**.

Shipping through a middle point like this is called **transshipment**.

In [10]:
HUB = "Long Beach hub"
to_hub = {"Vietnam": 1.4, "Indonesia": 1.5, "China": 1.2}             # $ per pair, factory -> hub
from_hub = {"Los Angeles": 0.3, "Portland": 0.6, "Memphis": 1.3,
            "New Jersey": 1.8, "Chicago": 1.5}                         # $ per pair, hub -> DC
hub_capacity = 30_000                                                   # pairs per week

via_hub = pd.DataFrame({d: {f: to_hub[f] + from_hub[d] for f in FACTORIES} for d in DCS})
saving = (ship_cost - via_hub).round(2)
saving.columns.name = "$ saved per pair via hub"
saving

$ saved per pair via hub,Los Angeles,Portland,Memphis,New Jersey,Chicago
Vietnam,0.3,0.3,0.7,0.7,0.6
Indonesia,0.4,0.4,0.5,0.3,0.4
China,0.3,0.2,1.1,1.2,1.0


**Every route is cheaper through the hub**, but by very different amounts. The biggest savings are China → New Jersey ($1.20), China → Memphis ($1.10) and China → Chicago ($1.00). The smallest are on the West Coast ($0.20 to $0.40), because LA and Portland are already close to the ports.

With only 30,000 pairs of hub space, we can't send everything through it. Which shipments deserve the hub? That's a job for the solver.

**What changes in the LP:**

| | New piece |
|---|---|
| New decision variables | Pairs from each factory **into** the hub (3), pairs **out of** the hub to each DC (5) |
| Objective | Add the cost of those new shipments |
| New constraint: **flow balance** | The hub doesn't make or store shoes, so **pairs in = pairs out** |
| New constraint: hub capacity | Pairs in ≤ 30,000 |
| Changed constraints | A factory's total now includes what it sends to the hub; a DC's total includes what it gets from the hub |

The flow balance rule is the important new idea. Every warehouse, port and DC in a real network has one: what arrives, minus what leaves, equals the change in stock.

In [11]:
def solve_with_hub(hub_capacity):
    model = pulp.LpProblem("shipping_with_hub", pulp.LpMinimize)

    # Decision variables
    ship = {(f, d): model.add_variable(var_name("ship", f, d), lowBound=0) for f in FACTORIES for d in DCS}
    into_hub = {f: model.add_variable(var_name("to_hub", f), lowBound=0) for f in FACTORIES}
    out_of_hub = {d: model.add_variable(var_name("from_hub", d), lowBound=0) for d in DCS}

    # Objective: direct + into-hub + out-of-hub shipping cost
    model += (pulp.lpSum(ship_cost.loc[f, d] * ship[f, d] for f in FACTORIES for d in DCS)
              + pulp.lpSum(to_hub[f] * into_hub[f] for f in FACTORIES)
              + pulp.lpSum(from_hub[d] * out_of_hub[d] for d in DCS))

    # A factory's direct shipments plus what it sends to the hub <= what it makes
    for f in FACTORIES:
        model += pulp.lpSum(ship[f, d] for d in DCS) + into_hub[f] <= supply[f], var_name("supply", f)

    # A DC's direct deliveries plus what it gets from the hub >= what it needs
    for d in DCS:
        model += pulp.lpSum(ship[f, d] for f in FACTORIES) + out_of_hub[d] >= demand[d], var_name("demand", d)

    # Flow balance: the hub stores nothing, so pairs in = pairs out
    model += pulp.lpSum(into_hub.values()) == pulp.lpSum(out_of_hub.values()), "hub_in_equals_out"

    # Hub capacity
    model += pulp.lpSum(into_hub.values()) <= hub_capacity, "hub_capacity"

    stats = model.solve(pulp.HiGHS(msg=False))
    assert stats.status == pulp.LpSolveStatus.Optimal, stats.status
    return model, ship, into_hub, out_of_hub


model_b, ship_b, into_hub_b, out_of_hub_b = solve_with_hub(hub_capacity)
cost_b = pulp.value(model_b.objective)
hub_value = abs(model_b.get_constraint_by_name("hub_capacity").pi)

print(f"Cost without hub: ${cost_a:,.0f}")
print(f"Cost with hub:    ${cost_b:,.0f}   -> hub saves ${cost_a - cost_b:,.0f} every week")
print(f"Hub used: {sum(v.varValue for v in into_hub_b.values()):,.0f} of {hub_capacity:,} pairs")
print(f"One more pair of hub capacity would save: ${hub_value:.2f}")

Cost without hub: $295,000
Cost with hub:    $274,500   -> hub saves $20,500 every week
Hub used: 30,000 of 30,000 pairs
One more pair of hub capacity would save: $0.60


In [12]:
flows = ([(f, d, q) for (f, d), v in ship_b.items() if (q := v.varValue) > 1e-6]
         + [(f, HUB, q) for f, v in into_hub_b.items() if (q := v.varValue) > 1e-6]
         + [(HUB, d, q) for d, v in out_of_hub_b.items() if (q := v.varValue) > 1e-6])
flow_picture(flows, f"Solver plan with the Long Beach hub: ${cost_b:,.0f}/week")

**Reading the plan:**
- The hub is **full** (30,000 of 30,000 pairs). It's used for **Memphis** (all 25,000 pairs) and part of **Chicago** (5,000), the far-inland DCs where the hub saves the most.
- **China** puts 10,000 pairs into the hub, because China → hub → Memphis saves $1.10 a pair. China still covers Portland directly.
- **Indonesia** still ships New Jersey directly. Its saving there via the hub is only $0.30, so the scarce hub space is worth more for other shipments.
- **LA and Portland** are still served directly. Their savings are the smallest, so they get no hub space.

**Why is one more pair of hub space worth exactly $0.60?** Vietnam has spare pairs. Vietnam → hub → Chicago costs $2.90, and Vietnam → Chicago directly costs $3.50. One more pair of hub space would move one Chicago pair onto that cheaper path: $3.50 − $2.90 = **$0.60**.

## 8. How much hub capacity is worth building?

Let's vary hub capacity from 0 to 105,000 pairs and watch the weekly cost and the value of one more pair of capacity.

In [13]:
from plotly.subplots import make_subplots

capacities = list(range(0, 105_001, 500))
costs, values = [], []
for cap in capacities:
    m_c, *_ = solve_with_hub(cap)
    costs.append(pulp.value(m_c.objective))
    values.append(abs(m_c.get_constraint_by_name("hub_capacity").pi))
# Exactly at a kink the solver may report either side's value; smooth those single points
for i in range(1, len(values) - 1):
    if abs(values[i] - values[i - 1]) > 1e-6 and abs(values[i] - values[i + 1]) > 1e-6:
        values[i] = values[i + 1]

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Weekly shipping cost vs hub capacity", "Saving from one more pair of hub capacity"))
fig.add_trace(go.Scatter(x=capacities, y=costs, mode="lines",
                         hovertemplate="%{x:,} pairs<br>cost $%{y:,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=capacities, y=values, mode="lines", line_shape="hv",
                         hovertemplate="%{x:,} pairs<br>saves $%{y:.2f} per pair<extra></extra>"), row=1, col=2)
for col in (1, 2):
    fig.add_vline(x=hub_capacity, line_dash="dot", line_color="gray", row=1, col=col)
fig.add_annotation(x=hub_capacity, y=max(costs), text="today: 30,000", showarrow=False, yshift=10, row=1, col=1)
fig.update_xaxes(title_text="Hub capacity (pairs per week)", tickformat=",")
fig.update_yaxes(title_text="Shipping cost", tickprefix="$", row=1, col=1)
fig.update_yaxes(title_text="$ saved per extra pair", tickprefix="$", row=1, col=2)
fig.update_layout(height=430, width=1000, showlegend=False, template="plotly_white")
fig.show()

breaks = [capacities[i] for i in range(1, len(capacities)) if abs(values[i] - values[i - 1]) > 1e-6]
pd.DataFrame(
    [{"hub capacity (pairs)": f"{lo:,} to {hi:,}", "saving per extra pair": f"${values[capacities.index(lo)]:.2f}"}
     for lo, hi in zip([0] + breaks, breaks + [capacities[-1]])]
).set_index("hub capacity (pairs)")

,saving per extra pair
hub capacity (pairs),
"0 to 25,000",$0.70
"25,000 to 35,000",$0.60
"35,000 to 40,000",$0.50
"40,000 to 85,000",$0.30
"85,000 to 100,000",$0.20
"100,000 to 105,000",$0.00


**Read the right-hand chart left to right:** each extra pair of hub space saves a bit less than the one before. The solver gives the hub to the shipments with the biggest savings first, so whatever capacity you add next goes to a smaller saving. Above **100,000 pairs** the extra space is worth **$0**, because that's all the DCs need: every pair already goes through the hub.

This pattern, called **diminishing returns**, shows up in almost every capacity question.

**How a manager uses this:** say adding hub space costs **$0.45 per pair per week** (lease, labor). Then expanding is worth it up to about **40,000 pairs**, where the saving drops to $0.30, below the $0.45 cost. The chart answers "how big should the hub be?" directly.

## 9. How does the solver actually find the answer?

You don't need the details for the interview. You do need the picture, because "explain simplex" is a common warm-up question.

- **Every rule cuts away some plans.** "Vietnam ships at most 50,000" rules out every plan where it ships more. Together, the rules leave a region of allowed plans, shaped like a cut gemstone with flat faces and sharp corners.
- **The best plan is always at a corner.** Cost is a straight-line (linear) function, so you can always lower it by sliding toward the edge of the allowed region, until you hit a corner.
- **The simplex method walks from corner to corner.** It starts at one corner, moves to a neighboring corner that's cheaper, and stops when no neighbor is cheaper. That last corner is the best plan.

**Why this explains what you saw:** a corner is a plan where as many things as possible are pushed to their limits, or to zero. That's why the solver used only **7 of 15 routes** and left the rest at exactly zero. A hand-made plan tends to spread shipments around; an LP answer is *sparse*.

(HiGHS also has *interior point* methods that cut through the middle of the region instead of walking the edges. They are faster on very large problems.)

## Recap: say these out loud

1. **What's the transportation problem?** Sources with limited supply, destinations with demand, a cost per route. Choose how much goes on each route to meet demand at the lowest cost.
2. **Why not ship on the cheapest route first?** It uses up cheap sources that another destination needs more. Here it cost **$6,000 a week** extra.
3. **What's a DC's shadow price?** What serving one more pair there would cost, including knock-on shuffles. New Jersey: **$3.70**, not the $3.60 cheapest route.
4. **What's a reduced cost?** How much cheaper an unused route must get before it's worth using. Vietnam → Portland: **$0.10**.
5. **What's transshipment?** Shipping through a middle point like a hub. It adds a **flow balance** rule: in = out.
6. **What's the hub worth?** It saves **$20,500 a week** at 30,000 pairs. One more pair of capacity is worth **$0.60**, falling as the hub gets bigger.
7. **How does simplex work?** The best plan is at a corner of the allowed region. Simplex walks from corner to cheaper corner until none is cheaper.

**Words from today**

| Word | Meaning |
|---|---|
| Transportation problem | Supply points → demand points, cost per route, minimize total cost |
| Shadow price (min problem) | Extra cost of 1 more unit of demand, or saving from 1 more unit of capacity |
| Reduced cost | How much an unused option's cost must drop before it's worth using |
| Tie / alternative optimum | Two different plans with the same lowest cost (reduced cost = $0 on an unused route) |
| Transshipment | Routing through an intermediate point (hub, port, cross-dock) |
| Flow balance | At a node: what comes in − what goes out = change in stock |
| Diminishing returns | Each extra unit of capacity is worth less than the last |
| Simplex | Solver method that walks from corner to corner of the allowed plans |

**Try it yourself:**
1. Set `supply["Indonesia"] = 20_000` and re-run Part A. *Predict first:* which DC loses its cheap source, and does New Jersey's shadow price go up?
2. Set `hub_capacity = 0` in Part B. You should get exactly the Part A cost. Why?